# Fleet sizing & route optimization: empirical analysis
**Question:** what is the smallest robot fleet that keeps every bin's wait under 15 minutes, and does OR-Tools routing actually beat the greedy dispatcher?

**Method:** simulate many days with different random seeds, compare policies on the *same* seeds, and report uncertainty (not single runs).

**Run it:** Run All. Set `QUICK = True` below for a ~2 min smoke test; the full run takes roughly 5-10 min.

> The simulator in section 2 is a **stand-in** so this notebook works today. When the lead's headless sim exists, wrap it with the same signature as `simulate()` and everything below still runs.

## 0. Install & imports

In [3]:
# Run once. If the import below still fails afterwards, restart the kernel.
!pip install --user ortools


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


In [2]:
import sys, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for p in ("app", "../app", "waste-fleet/app", "."):
    if os.path.exists(os.path.join(p, "optimize.py")):
        sys.path.insert(0, p)
        break
else:
    raise FileNotFoundError("optimize.py not found - put it next to this notebook or in ./app")

from optimize import plan_routes, fleet_size_sweep, fleet_size_study

# ---- knobs ----
QUICK = False                 # True = fast smoke run
TARGET_MAX_WAIT = 15.0        # service target: no bin waits more than 15 min
SIZES = list(range(2, 9))
N_SEEDS = 3 if QUICK else 8
SENS_SEEDS = 3 if QUICK else 5
PLAN_TIME_S = 0.05            # OR-Tools time limit per dispatch call
BALANCE_WEIGHT = 10           # extra penalty on the longest route in the 'balanced' policy
POLICIES = ("greedy", "optimized", "balanced")
SEEDS = list(range(N_SEEDS))
plt.rcParams["figure.figsize"] = (9, 5)

ModuleNotFoundError: No module named 'numpy'

## 1. Contract smoke test
Checks `plan_routes` on the Contract 2 example: every stop assigned, no robot over capacity.

In [ ]:
test_nodes = ["dock", "a", "b", "c"]
test_tt = [[0, 4, 6, 8], [4, 0, 3, 7], [6, 3, 0, 10], [8, 7, 10, 0]]
test_robots = [{"id": f"r{i}", "start": "dock", "end": "dock",
                "capacity_left": 60, "max_minutes": 40} for i in (1, 2)]
test_stops = [{"bin_id": f"b{i}", "node": n, "demand": 30} for i, n in zip((1, 2, 3), "abc")]

out = plan_routes(test_nodes, test_tt, test_robots, test_stops, time_limit_s=0.5)
print(out)
assert not out["unassigned"], "every stop should be assigned"
assert sorted(b for r in out["routes"].values() for b in r) == ["b1", "b2", "b3"]
assert all(len(r) * 30 <= 60 for r in out["routes"].values()), "capacity respected"
print("contract example OK")

## 2. Stand-in simulator
A fixed 12-node map (node 0 = dock, 11 bins), bin fill rates that follow a time-of-day curve (class breaks + lunch peak) with noise, and robots with capacity and battery limits. Each **seed** changes the day's demand noise, so seeds are independent simulated days.

Three dispatch policies:
- **greedy**: the nearest feasible idle robot takes one bin at a time (the spec's version 1).
- **optimized**: every 3 sim-minutes, `plan_routes` plans multi-stop routes for idle robots over bins at 80%+ plus nearby bins at 60%+, minimizing total travel (the spec's version 2).
- **balanced**: same, but `plan_routes(..., balance_weight=BALANCE_WEIGHT)` also penalizes the longest route. *Hypothesis to test:* minimizing total travel tends to chain bins onto one robot, which can lengthen waits; balancing may fix that. The data decides.

In [ ]:
# ---- fixed world (same for every seed) ----
_layout = np.random.default_rng(7)
N_NODES = 12                                    # node 0 is the dock
XY = _layout.uniform(0, 900, size=(N_NODES, 2)) # meters
SPEED_M_PER_MIN = 130.0
T = np.ceil(np.linalg.norm(XY[:, None] - XY[None], axis=2) / SPEED_M_PER_MIN).astype(int)  # minutes
NODE_NAMES = [f"n{i}" for i in range(N_NODES)]
N_BINS = N_NODES - 1                            # bin i sits on node i + 1
BASE_RATE = _layout.uniform(20, 48, size=N_BINS)  # % fill per hour at curve value 1.0

DAY_MIN, START_HOUR = 720, 8                    # simulate 8:00-20:00
CAP, BIN_SHARE = 100.0, 0.4                     # robot capacity; a full bin = 40% of it
BATTERY_MIN, UNLOAD_MIN = 120, 5                # minutes of travel per charge; dock dwell
DISPATCH_EVERY, NEARBY_MIN = 3, 10

def demand_curve(hour):
    return (0.25 + 1.2 * math.exp(-((hour - 12.5) / 1.2) ** 2)      # lunch peak
            + 0.6 * math.exp(-((hour - 10.0) / 0.6) ** 2)           # class break
            + 0.6 * math.exp(-((hour - 15.0) / 0.7) ** 2))          # class break

CURVE = np.array([demand_curve(START_HOUR + t / 60) for t in range(DAY_MIN)])


def simulate(n_robots, seed, policy="optimized", demand_scale=1.0, planner=None, plan_time_s=None):
    """One simulated day -> metrics. policy: "greedy", "optimized" or "balanced"."""
    planner = planner or plan_routes
    plan_time_s = PLAN_TIME_S if plan_time_s is None else plan_time_s
    rng = np.random.default_rng(seed)
    day_mult = rng.lognormal(0, 0.2, N_BINS)                         # each bin's busy-ness today
    noise = np.clip(1 + 0.3 * rng.standard_normal((DAY_MIN, N_BINS)), 0, None)
    fill = rng.uniform(0, 50, N_BINS)
    node_of = np.arange(1, N_BINS + 1)
    assigned = [False] * N_BINS
    t80 = [None] * N_BINS                                            # minute a bin crossed 80%
    robots = [dict(id=k, loc=0, state="idle", arrive=0, dest=0, target=None,
                   route=[], load=0.0, batt=float(BATTERY_MIN)) for k in range(n_robots)]
    waits, overflow, distance, pickups = [], 0, 0, 0

    def go(r, node, t, target):
        nonlocal distance
        dur = int(T[r["loc"]][node])
        r["batt"] -= dur
        distance += dur
        r.update(state="moving", arrive=t + dur, dest=node, target=target)

    def to_dock(r, t):
        go(r, 0, t, ("dock",))

    def next_leg(r, t):
        if r["route"]:
            i = r["route"].pop(0)
            go(r, int(node_of[i]), t, ("bin", i))
        elif policy in ("optimized", "balanced") or r["load"] > 85 or r["batt"] < 0.25 * BATTERY_MIN:
            to_dock(r, t)
        else:
            r["state"], r["target"] = "idle", None

    def arrive(r, t):
        nonlocal pickups
        r["loc"] = r["dest"]
        kind = r["target"][0]
        if kind == "bin":
            i = r["target"][1]
            d = fill[i] * BIN_SHARE
            if r["load"] + d <= CAP:
                if t80[i] is not None:
                    waits.append(t - t80[i])
                r["load"] += d
                fill[i], t80[i] = 0.0, None
                pickups += 1
                assigned[i] = False
                next_leg(r, t)
            else:                          # bin outgrew the robot's space: drop the rest of the route
                assigned[i] = False
                for j in r["route"]:
                    assigned[j] = False
                r["route"] = []
                to_dock(r, t)
        elif kind == "dock":               # unload and recharge
            r["load"], r["batt"] = 0.0, float(BATTERY_MIN)
            r.update(state="moving", arrive=t + UNLOAD_MIN, dest=0, target=("ready",))
        else:
            r["state"], r["target"] = "idle", None

    for t in range(DAY_MIN):
        fill += BASE_RATE * day_mult * CURVE[t] * demand_scale * noise[t] / 60
        np.minimum(fill, 100.0, out=fill)
        for i in range(N_BINS):
            if fill[i] >= 80 and t80[i] is None:
                t80[i] = t
        overflow += int((fill >= 100).sum())                         # bin-minutes at 100%
        for r in robots:
            if r["state"] == "moving" and r["arrive"] <= t:
                arrive(r, t)

        if t % DISPATCH_EVERY:
            continue
        idle = [r for r in robots if r["state"] == "idle"]
        for r in list(idle):                                         # full or low battery -> dock
            if r["loc"] != 0 and (r["load"] > 85 or r["batt"] < 0.25 * BATTERY_MIN):
                to_dock(r, t)
                idle.remove(r)
        open_ = [i for i in range(N_BINS) if fill[i] >= 80 and not assigned[i]]
        if not open_ or not idle:
            continue

        if policy == "greedy":
            for i in sorted(open_, key=lambda i: -fill[i]):
                d, n = fill[i] * BIN_SHARE, int(node_of[i])
                feas = [r for r in idle if r["load"] + d <= CAP and T[r["loc"]][n] + T[n][0] <= r["batt"]]
                if not feas:
                    continue
                r = min(feas, key=lambda r: T[r["loc"]][n])
                idle.remove(r)
                assigned[i] = True
                r["route"] = [i]
                next_leg(r, t)
            if any(not assigned[i] for i in open_):                  # leftovers: free up loaded robots
                for r in idle:
                    if r["loc"] != 0 and r["load"] > 0:
                        to_dock(r, t)
        else:
            def near(i):
                return any(T[int(node_of[i])][int(node_of[j])] <= NEARBY_MIN for j in open_)
            cand = open_ + [i for i in range(N_BINS)
                            if 60 <= fill[i] < 80 and not assigned[i] and near(i)]
            robots_in = [{"id": f"r{r['id']}", "start": NODE_NAMES[r["loc"]], "end": NODE_NAMES[0],
                          "capacity_left": CAP - r["load"], "max_minutes": float(r["batt"])}
                         for r in idle]
            stops_in = [{"bin_id": f"b{i}", "node": NODE_NAMES[int(node_of[i])],
                         "demand": float(fill[i] * BIN_SHARE)} for i in cand]
            kw = {"balance_weight": BALANCE_WEIGHT} if policy == "balanced" else {}
            plan = planner(NODE_NAMES, T.tolist(), robots_in, stops_in, time_limit_s=plan_time_s, **kw)
            by_id = {f"r{r['id']}": r for r in idle}
            for rid, route in plan["routes"].items():
                if route:
                    r = by_id[rid]
                    r["route"] = [int(b[1:]) for b in route]
                    for i in r["route"]:
                        assigned[i] = True
                    next_leg(r, t)

    for i in range(N_BINS):                                          # bins still waiting at close
        if t80[i] is not None:
            waits.append(DAY_MIN - t80[i])
    return {"avg_wait_min": float(np.mean(waits)) if waits else 0.0,
            "max_wait_min": float(max(waits)) if waits else 0.0,
            "overflow_minutes": float(overflow),
            "distance_min": float(distance), "pickups": pickups}

### Sanity check: the demand the robots face
Should show the lunch peak and two class-break bumps.

In [ ]:
hours = START_HOUR + np.arange(DAY_MIN) / 60
total_rate = BASE_RATE.sum() * CURVE / 100            # bins' worth of fill per hour, whole campus
plt.plot(hours, total_rate)
plt.xlabel("hour of day"); plt.ylabel("bins filled per hour (all bins)")
plt.title("Simulated campus-wide demand"); plt.show()
print("one example day:", simulate(4, 0, "greedy"))

## 3. Experiment 1: how many robots? (multi-seed)
Every fleet size, 2 to 8, runs on the **same seeds** for both policies, so differences come from the policy and fleet size, not luck. A single seed gives one noisy answer; many seeds tell us on what share of days the target is met.

In [ ]:
def run_grid(policy, sizes, seeds, demand_scale=1.0):
    rows = []
    for n in sizes:
        for s in seeds:
            rows.append({"policy": policy, "demand_scale": demand_scale, "robots": n, "seed": s,
                         **simulate(n, s, policy, demand_scale)})
        print(f"  {policy:9s} scale {demand_scale} robots {n} done")
    return pd.DataFrame(rows)

runs = pd.concat([run_grid(p, SIZES, SEEDS) for p in POLICIES], ignore_index=True)
os.makedirs("results", exist_ok=True)
runs.to_csv("results/runs_main.csv", index=False)       # raw per-seed data = reproducible
print(len(runs), "simulated days saved to results/runs_main.csv")

In [ ]:
def summarize_runs(df):
    return (df.groupby(["policy", "demand_scale", "robots"])
              .agg(avg_wait=("avg_wait_min", "mean"),
                   max_wait_mean=("max_wait_min", "mean"),
                   max_wait_p95=("max_wait_min", lambda s: s.quantile(0.95)),
                   overflow=("overflow_minutes", "mean"),
                   share_ok=("max_wait_min", lambda s: (s <= TARGET_MAX_WAIT).mean()),
                   days=("seed", "count"))
              .reset_index())

def recommended(summary, policy, scale=1.0, share=0.95):
    s = summary[(summary.policy == policy) & (summary.demand_scale == scale) & (summary.share_ok >= share)]
    return int(s.robots.min()) if len(s) else None

summ = summarize_runs(runs)
summ[summ.demand_scale == 1.0].round(2)

In [ ]:
# Contract functions on the cached per-seed results (no re-simulation)
cache = runs.set_index(["policy", "demand_scale", "robots", "seed"]).to_dict("index")
cached = lambda policy, scale=1.0: (lambda n, s: cache[(policy, scale, n, s)])

rec_by_policy = {p: recommended(summ, p) for p in POLICIES}
best = min(POLICIES, key=lambda p: rec_by_policy[p] or 99)      # policy needing the smallest fleet
print("fleet needed per policy (target met on >=95% of days):",
      {p: (n if n else f">{max(SIZES)}") for p, n in rec_by_policy.items()})
print("using", best, "for the contract functions below\n")

single = fleet_size_sweep(cached(best), sizes=SIZES, target_max_wait_min=TARGET_MAX_WAIT, seed=0)
study = fleet_size_study(cached(best), sizes=SIZES, seeds=SEEDS, target_max_wait_min=TARGET_MAX_WAIT)
print("one seed says  :", single["recommended"], "robots")
print("multi-seed says:", study["recommended"], f"robots (target met on >=95% of {N_SEEDS} days)")
pd.DataFrame(study["results"]).round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
colors = {"greedy": "tab:orange", "optimized": "tab:blue", "balanced": "tab:green"}
for pol in POLICIES:
    d = runs[(runs.policy == pol) & (runs.demand_scale == 1.0)]
    g = d.groupby("robots")["max_wait_min"]
    ax[0].plot(g.mean().index, g.mean().values, marker="o", color=colors[pol], label=pol)
    ax[0].fill_between(g.mean().index, g.quantile(0.05).values, g.quantile(0.95).values,
                       color=colors[pol], alpha=0.12)
    o = d.groupby("robots")["overflow_minutes"].mean()
    ax[1].plot(o.index, o.values, marker="o", color=colors[pol], label=pol)
ax[0].axhline(TARGET_MAX_WAIT, ls="--", color="gray", label=f"{TARGET_MAX_WAIT:.0f}-min target")
ax[0].set(xlabel="robots", ylabel="worst wait of the day (min)",
          title="Worst wait vs fleet size\n(line = mean, band = 5th-95th percentile across days)")
ax[1].set(xlabel="robots", ylabel="bin-minutes at 100% full", title="Overflow vs fleet size (mean)")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

## 4. Experiment 2: does optimization beat greedy? (paired comparison)
At a fixed fleet size, every policy faces the identical day (same seed), so we look at the **per-day difference from greedy** (negative = better than greedy), with a bootstrap 95% confidence interval. If the interval excludes 0, the difference is unlikely to be luck.

In [ ]:
K = rec_by_policy[best] or SIZES[len(SIZES) // 2]
base = runs[(runs.policy == "greedy") & (runs.robots == K) & (runs.demand_scale == 1.0)].set_index("seed")
per_pickup = lambda d: d.distance_min / d.pickups.replace(0, np.nan)

def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x.dropna()); rng = np.random.default_rng(seed)
    return np.percentile(rng.choice(x, size=(n, len(x))).mean(axis=1), [2.5, 97.5])

rows, paired_all = [], {}
for pol in ("optimized", "balanced"):
    other = runs[(runs.policy == pol) & (runs.robots == K) & (runs.demand_scale == 1.0)].set_index("seed")
    paired = pd.DataFrame({
        "max_wait_min": other.max_wait_min - base.max_wait_min,
        "avg_wait_min": other.avg_wait_min - base.avg_wait_min,
        "overflow_minutes": other.overflow_minutes - base.overflow_minutes,
        "travel_min_per_pickup": per_pickup(other) - per_pickup(base)})
    paired_all[pol] = paired
    for col in paired:
        lo, hi = boot_ci(paired[col])
        rows.append({"policy": pol, "metric": col, "mean_diff": paired[col].mean(),
                     "ci95_low": lo, "ci95_high": hi, "days_better": (paired[col] < 0).mean()})
cmp_table = pd.DataFrame(rows)
print(f"Paired differences vs greedy at {K} robots over {len(base)} days (negative = better than greedy)")
cmp_table.round(2)

In [ ]:
metrics = ["max_wait_min", "avg_wait_min", "overflow_minutes", "travel_min_per_pickup"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax_, col in zip(axes, metrics):
    ax_.boxplot([paired_all[p][col].dropna() for p in ("optimized", "balanced")])
    ax_.set_xticklabels(["optimized", "balanced"])
    ax_.axhline(0, color="gray", ls="--"); ax_.set_title(col, fontsize=10)
fig.suptitle(f"Per-day difference from greedy at {K} robots (below 0 = better than greedy)")
plt.tight_layout(); plt.show()

## 5. Experiment 3: sensitivity to demand
What if we mis-estimated demand? Re-run the best policy from Experiment 1 with demand scaled to 0.7x (slow day) and 1.3x (finals week), and see how the required fleet changes. (Fewer seeds here to save time, so treat the 95% rule as coarse.)

In [ ]:
sens_seeds = SEEDS[:SENS_SEEDS]
parts = [runs[(runs.policy == best) & (runs.demand_scale == 1.0) & runs.seed.isin(sens_seeds)]]
for sc in (0.7, 1.3):
    parts.append(run_grid(best, SIZES, sens_seeds, demand_scale=sc))
sens = pd.concat(parts, ignore_index=True)
sens.to_csv("results/runs_sensitivity.csv", index=False)
sens_summ = summarize_runs(sens)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for sc in (0.7, 1.0, 1.3):
    d = sens_summ[sens_summ.demand_scale == sc]
    ax[0].plot(d.robots, d.share_ok, marker="o", label=f"{sc}x demand")
ax[0].axhline(0.95, ls="--", color="gray")
ax[0].set(xlabel="robots", ylabel=f"share of days meeting {TARGET_MAX_WAIT:.0f}-min target",
          title=f"Service level vs fleet size ({best} policy)")
ax[0].legend()
need = [recommended(sens_summ, best, sc) for sc in (0.7, 1.0, 1.3)]
ax[1].bar(["0.7x", "1.0x", "1.3x"], [n or max(SIZES) + 1 for n in need], color="tab:blue")
for i, n in enumerate(need):
    ax[1].text(i, (n or max(SIZES) + 1) + 0.1, str(n) if n else f">{max(SIZES)}", ha="center")
ax[1].set(xlabel="demand level", ylabel="robots needed", title="Required fleet vs demand")
plt.tight_layout(); plt.show()

## 6. Results summary
Prints the headline numbers for the slides, straight from the data.

In [ ]:
fmt = lambda n: str(n) if n else f">{max(SIZES)}"
print(f"Fleet needed to keep every bin under {TARGET_MAX_WAIT:.0f} min on >=95% of simulated days:")
for p in POLICIES:
    print(f"  {p:10s}: {fmt(rec_by_policy[p])} robots")
for pol in ("optimized", "balanced"):
    print(f"\n{pol} vs greedy at {K} robots (paired over {len(base)} days, 95% bootstrap CI):")
    for _, r in cmp_table[cmp_table.policy == pol].iterrows():
        verdict = ("clearly better" if r.ci95_high < 0
                   else "clearly worse" if r.ci95_low > 0 else "no clear difference")
        print(f"  {r.metric:22s} {r.mean_diff:+8.2f}  [{r.ci95_low:+.2f}, {r.ci95_high:+.2f}]  -> {verdict}")
print(f"\nDemand sensitivity for '{best}' (robots needed at 0.7x / 1.0x / 1.3x): {' / '.join(fmt(n) for n in need)}")

## Limitations (say these out loud; it builds trust)
- Demand is **simulated** (a time-of-day curve plus noise), not measured; real bin sensors would replace it.
- This notebook uses a stand-in simulator; numbers should be regenerated with the lead's sim before the final slides.
- Waits count only bins that crossed 80%; bins collected early at 60-80% by the optimizer count as no wait, which slightly favors it on average wait (max wait and overflow are not affected).
- The solver stops on a time limit, so rerunning can change results slightly; raw per-day data is saved in `results/` for reproducibility.